# Causal Language Modeling Practice

**Topic:** Transformer Architectures &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X_train` `(8000, 6)`, `y_train` `(8000,)`: contexts of 6 tokens (vocabulary 0–11) and the token that came next.
- `X_test` `(2000, 6)`: predict the next token. `y_test` is used by the evaluation cell.
- Hidden rule: with prob 0.85, `next = permutation[context[some fixed position]]`. Otherwise it is uniform random. The best possible accuracy is therefore about 0.86.

## Your task

1. Establish the bigram baseline (predict from `context[-1]`) and watch it fail.
2. Build a model that can exploit the right position: a small causal transformer LM in PyTorch (embedding + attention + head trained on the contexts), or a direct per-offset analysis if you first want to *find* the dependency.
3. Set `predictions` to your `(2000,)` predicted next tokens.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell (~0.86 is optimal).


In [ ]:
import numpy as np

rng = np.random.default_rng(481)
VOCAB = 12
_perm = rng.permutation(VOCAB)          # the hidden transition rule

def make(n, seed):
    r = np.random.default_rng(seed)
    X = r.integers(0, VOCAB, (n, 6))
    determined = r.random(n) < 0.85
    y = np.where(determined, _perm[X[:, -3]], r.integers(0, VOCAB, n))
    return X, y

X_train, y_train = make(8000, 491)
X_test, y_test = make(2000, 492)
print(f"X_train {X_train.shape}, vocab {VOCAB}: which context position matters?")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.75

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {int(_cls):>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

Find the dependency first: for each context offset, measure how well `next` is a deterministic function of that token on the training set. Offset -3 wins by a mile, so learn its token-to-token mapping by majority vote and predict with it, exactly the circuit a small causal transformer would learn.

```python
import numpy as np

# Which context position predicts y? Score each offset by majority-vote accuracy.
def offset_accuracy(col):
    table = np.zeros(VOCAB, dtype=int)
    for t in range(VOCAB):
        rows = y_train[X_train[:, col] == t]
        table[t] = np.bincount(rows.astype(np.intp), minlength=VOCAB).argmax()
    return float((table[X_train[:, col]] == y_train).mean()), table

baseline_acc, _ = offset_accuracy(5)   # column 5 is context[-1], the bigram baseline
print(f"bigram baseline train accuracy: {baseline_acc:.3f}")

scores = [offset_accuracy(c) for c in range(6)]
best_col = int(np.argmax([s for s, _ in scores]))
acc, table = scores[best_col]
print(f"best context position: index {best_col}, which is offset {best_col - 6} (train accuracy {acc:.3f})")

predictions = table[X_test[:, best_col]]
```
